# 🎯 Bước 4 & 5: Lọc Subset Đại diện, Làm sạch, Join 6 Bảng & Xuất Parquet
**Dự án:** Corporación Favorita Grocery Sales Forecasting  
**Thành viên:** Thành — Data & Infrastructure Engineer  
**Mục tiêu:**  
1. Lọc Subset đại diện 2 chiều (Thời gian: 2016–2017 & Không gian: 10 cửa hàng Stratified).
2. Làm sạch dữ liệu: xử lý hàng trả lại (unit_sales < 0), null onpromotion, Forward Fill giá dầu.
3. Join 6 bảng dữ liệu thành một DataFrame thống nhất.
4. Xuất file Parquet mẫu (Sample 100k dòng bàn giao cho Tiến) và Full Subset (~10-15M dòng cho pipeline chung).

### 1. Khởi tạo Spark Session kết nối Cụm Master

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Subset_and_Join_Pipeline") \
    .master("spark://spark-master:7077") \
    .config("spark.driver.host", "jupyter-pyspark") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.executor.memory", "2g") \
    .config("spark.executor.cores", "2") \
    .config("spark.sql.shuffle.partitions", "16") \
    .getOrCreate()

print("✅ Spark Session Connected successfully!")
print(f"Spark version: {spark.version}")

✅ Spark Session Connected successfully!
Spark version: 3.5.0


### 2. Thiết lập Chiến lược Lọc Subset Đại diện
- **Lọc theo Thời gian:** 01/01/2016 đến 15/08/2017 (loại bỏ nhiễu và giai đoạn khuyết onpromotion).
- **Lọc theo Không gian:** 10 cửa hàng đại diện cho 5 Types (A, B, C, D, E) và đa dạng vị trí địa lý (Quito, Guayaquil, thị trấn nhỏ, vùng biển).

In [2]:
# Danh sách 10 cửa hàng đại diện (được chọn từ phân tích EDA Bước 2)
selected_stores = [
    44,  # Type A, Cluster 5  — Quito (thủ đô)
    52,  # Type A, Cluster 11 — Manta (thành phố ven biển)
    11,  # Type B, Cluster 6  — Cayambe
    31,  # Type B, Cluster 10 — Babahoyo
    30,  # Type C, Cluster 3  — Guayaquil (đô thị lớn)
    14,  # Type C, Cluster 7  — Riobamba
    3,   # Type D, Cluster 8  — Quito
    25,  # Type D, Cluster 1  — Salinas (thị trấn ven biển)
    28,  # Type E, Cluster 10 — Guayaquil
    43   # Type E, Cluster 10 — Esmeraldas
]

print(f"Selected {len(selected_stores)} stores: {selected_stores}")

Selected 10 stores: [44, 52, 11, 31, 30, 14, 3, 25, 28, 43]


### 3. Đọc, Lọc và Làm sạch Bảng Giao dịch Chính (`train.csv`)

In [3]:
# Đọc train.csv
train_raw = spark.read.csv("/home/jovyan/data/raw/train.csv", header=True, inferSchema=True)

# Chuyển date về DateType chuẩn
train_raw = train_raw.withColumn("date", F.to_date("date", "yyyy-MM-dd"))

# Lọc 2 chiều (Thời gian + 10 Cửa hàng)
train_subset = train_raw.filter(
    (F.col("date") >= "2016-01-01") &
    (F.col("date") <= "2017-08-15") &
    (F.col("store_nbr").isin(selected_stores))
)

# Làm sạch (Cleaning):
# 1. unit_sales < 0 là hàng bị trả lại -> đưa về 0 để dự báo nhu cầu mua
# 2. onpromotion null -> đưa về False
train_clean = train_subset.withColumn(
    "unit_sales",
    F.when(F.col("unit_sales") < 0, 0.0).otherwise(F.col("unit_sales"))
).withColumn(
    "onpromotion",
    F.when(F.col("onpromotion").isNull(), False).otherwise(F.col("onpromotion"))
)

subset_count = train_clean.count()
print(f"✅ Subset sau khi lọc & làm sạch: {subset_count:,} dòng")
train_clean.show(5)

✅ Subset sau khi lọc & làm sạch: 10,075,199 dòng
+--------+----------+---------+--------+----------+-----------+
|      id|      date|store_nbr|item_nbr|unit_sales|onpromotion|
+--------+----------+---------+--------+----------+-----------+
|66458908|2016-01-01|       25|  105574|      12.0|      false|
|66458909|2016-01-01|       25|  105575|       9.0|      false|
|66458910|2016-01-01|       25|  105857|       3.0|      false|
|66458911|2016-01-01|       25|  108634|       3.0|      false|
|66458912|2016-01-01|       25|  108701|       2.0|       true|
+--------+----------+---------+--------+----------+-----------+
only showing top 5 rows



### 4. Đọc & Xử lý 5 Bảng Phụ trợ
- **Oil:** Dùng Window function Forward Fill để bù giá dầu cho các ngày cuối tuần/ngày lễ.
- **Holidays:** Lọc các ngày lễ thực sự diễn ra (`transferred == False`), loại bỏ trùng lặp ngày.
- **Stores, Items, Transactions:** Chuẩn hóa tên cột để tránh trùng lặp sau khi join.

In [4]:
# 1. STORES
stores = spark.read.csv("/home/jovyan/data/raw/stores.csv", header=True, inferSchema=True) \
    .withColumnRenamed("type", "store_type")

# 2. ITEMS
items = spark.read.csv("/home/jovyan/data/raw/items.csv", header=True, inferSchema=True)

# 3. OIL (Giá dầu thô - Forward Fill)
oil = spark.read.csv("/home/jovyan/data/raw/oil.csv", header=True, inferSchema=True) \
    .withColumn("date", F.to_date("date", "yyyy-MM-dd"))

# Window function sắp xếp theo ngày để Forward Fill giá trị trước đó
oil_window = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, 0)
oil_clean = oil.withColumn(
    "oil_price",
    F.last(F.col("dcoilwtico"), ignorenulls=True).over(oil_window)
).drop("dcoilwtico")

# 4. HOLIDAYS (Ngày lễ)
holidays = spark.read.csv("/home/jovyan/data/raw/holidays_events.csv", header=True, inferSchema=True) \
    .withColumn("date", F.to_date("date", "yyyy-MM-dd"))

holidays_clean = holidays.filter(
    (F.col("transferred") == False) | (F.col("transferred").isNull())
).select(
    "date",
    F.lit(True).alias("is_holiday"),
    F.col("type").alias("holiday_type")
).dropDuplicates(["date"])

# 5. TRANSACTIONS
transactions = spark.read.csv("/home/jovyan/data/raw/transactions.csv", header=True, inferSchema=True) \
    .withColumn("date", F.to_date("date", "yyyy-MM-dd"))

print("✅ Đã xử lý và chuẩn hóa 5 bảng phụ trợ thành công!")

✅ Đã xử lý và chuẩn hóa 5 bảng phụ trợ thành công!


### 5. Join 6 Bảng thành 1 DataFrame Thống nhất

In [5]:
# Join 1: train_clean + stores (trên store_nbr)
df = train_clean.join(stores, on="store_nbr", how="inner")

# Join 2: + items (trên item_nbr)
df = df.join(items, on="item_nbr", how="inner")

# Join 3: + oil (trên date) - Left Join
df = df.join(oil_clean, on="date", how="left")

# Join 4: + holidays (trên date) - Left Join
df = df.join(holidays_clean, on="date", how="left")
df = df.withColumn("is_holiday", F.when(F.col("is_holiday").isNull(), False).otherwise(True))

# Join 5: + transactions (trên [date, store_nbr]) - Left Join
df = df.join(transactions, on=["date", "store_nbr"], how="left")
df = df.withColumn("transactions", F.when(F.col("transactions").isNull(), 0).otherwise(F.col("transactions")))

print("--- SCHEMA DATAFRAME SAU KHI JOIN 6 BẢNG ---")
df.printSchema()
print(f"Tổng số cột: {len(df.columns)}")
df.show(5)

--- SCHEMA DATAFRAME SAU KHI JOIN 6 BẢNG ---
root
 |-- date: date (nullable = true)
 |-- store_nbr: integer (nullable = true)
 |-- item_nbr: integer (nullable = true)
 |-- id: integer (nullable = true)
 |-- unit_sales: double (nullable = true)
 |-- onpromotion: boolean (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- store_type: string (nullable = true)
 |-- cluster: integer (nullable = true)
 |-- family: string (nullable = true)
 |-- class: integer (nullable = true)
 |-- perishable: integer (nullable = true)
 |-- oil_price: double (nullable = true)
 |-- is_holiday: boolean (nullable = false)
 |-- holiday_type: string (nullable = true)
 |-- transactions: integer (nullable = true)

Tổng số cột: 17
+----------+---------+--------+--------+----------+-----------+-------+-----------+----------+-------+---------+-----+----------+---------+----------+------------+------------+
|      date|store_nbr|item_nbr|      id|unit_sales|onpromotion|   city

### 6. Xuất Dữ liệu ra Định dạng Parquet (Bước 5)
- **Sample Parquet (100k dòng):** Xuất ngay để gửi cho Tiến viết Feature Engineering.
- **Full Subset Parquet:** Xuất toàn bộ subset đã làm sạch vào `data/processed/` phục vụ các thí nghiệm mở rộng và huấn luyện mô hình.

In [6]:
# ----- 6.1 XUẤT SAMPLE 100K DÒNG BÀN GIAO CHO TIẾN -----
sample_fraction = min(1.0, 100000.0 / subset_count)
sample_df = df.sample(fraction=sample_fraction, seed=42).limit(100000)

sample_path = "/home/jovyan/data/sample/favorita_sample_100k.parquet"
sample_df.coalesce(1).write.mode("overwrite").parquet(sample_path)
print(f"✅ ĐÃ XUẤT SAMPLE: {sample_path}")

# ----- 6.2 XUẤT FULL SUBSET (Phân vùng theo kích thước tối ưu) -----
full_path = "/home/jovyan/data/processed/favorita_full_subset.parquet"
# Chia thành 8-12 partitions để ghi song song trên cụm 3 workers
df.repartition(10).write.mode("overwrite").parquet(full_path)
print(f"✅ ĐÃ XUẤT FULL SUBSET: {full_path}")

✅ ĐÃ XUẤT SAMPLE: /home/jovyan/data/sample/favorita_sample_100k.parquet
✅ ĐÃ XUẤT FULL SUBSET: /home/jovyan/data/processed/favorita_full_subset.parquet


### 7. Kiểm tra Lại Dữ liệu Parquet đã Xuất & Giải phóng Tài nguyên Cụm

In [7]:
# Đọc kiểm tra sample
check_sample = spark.read.parquet("/home/jovyan/data/sample/favorita_sample_100k.parquet")
print(f"Sample row count: {check_sample.count():,}")

# Đọc kiểm tra full
check_full = spark.read.parquet("/home/jovyan/data/processed/favorita_full_subset.parquet")
print(f"Full subset row count: {check_full.count():,}")
print("Sample record:")
check_full.show(3)

# DỪNG SPARK SESSION ĐỂ GIẢI PHÓNG CORES CHO CÁC THÍ NGHIỆM TIẾP THEO
spark.stop()
print("✅ Spark Session stopped cleanly! Cores and memory released on cluster.")

Sample row count: 99,964
Full subset row count: 10,075,199
Sample record:
+----------+---------+--------+--------+----------+-----------+----------+----------+----------+-------+---------+-----+----------+---------+----------+------------+------------+
|      date|store_nbr|item_nbr|      id|unit_sales|onpromotion|      city|     state|store_type|cluster|   family|class|perishable|oil_price|is_holiday|holiday_type|transactions|
+----------+---------+--------+--------+----------+-----------+----------+----------+----------+-------+---------+-----+----------+---------+----------+------------+------------+
|2016-04-16|       28| 1695972|76462579|    25.814|      false| Guayaquil|    Guayas|         E|     10|  PRODUCE| 2014|         1|     NULL|      true|       Event|        1321|
|2016-05-04|        3|  220435|78156219|      65.0|      false|     Quito| Pichincha|         D|      8|GROCERY I| 1080|         0|    43.77|      true|       Event|        3271|
|2016-05-01|       43|  638978|